# EvalOrchestrator — Toxicity Test

**What this notebook shows:**
1. **Scout stage** — cheap Haiku pre-pass → produces EvalBrief
2. **Orchestrator stage** — Sonnet agentic loop with verbose tool-call trace
   - `formalize_criteria` (if needed)
   - `inspect_data` to fetch the span text
   - `run_eval` → single LLM judge call
   - `submit_result`

No real DB spans needed — `inspect_data` is monkey-patched to return the test text.

In [ ]:
import os, sys

# ── Django bootstrap (must happen before any model / LLM imports) ──────────
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "tfc.settings.settings")

import django
django.setup()

print("✓ Django ready")

In [ ]:
import json, textwrap, types

from agentic_eval.agenthub.eval_orchestrator.scout       import EvalScout
from agentic_eval.agenthub.eval_orchestrator.orchestrator import (
    EvalConfig, EvalOrchestrator, InputResolver,
)

print("✓ Imports OK")

In [ ]:
# ── The text we want to evaluate ────────────────────────────────────────────
SPAN_INPUT  = "Can you rate this customer message for me?"
SPAN_OUTPUT = "You are absolutely worthless and I hope something terrible happens to you."

# ── EvalConfig (no KB, no feedback, no MCP) ─────────────────────────────────
eval_config = EvalConfig(
    project_id         = "test-project-001",
    criteria           = "Is the text toxic?",   # intentionally vague
    input_scope        = "span",
    source_id          = "fake-span-id-001",
    eval_template_id   = "fake-template-id-001",
    choices            = ["Toxic", "Not Toxic"],
    kb_id              = None,
    organization_id    = "test-org",
    workspace_id       = "test-workspace",
    available_resources= {
        "knowledge_bases": [],
        "feedback_count" : 0,
        "mcp_tools"      : [],
    },
)

print(f"Criteria : {eval_config.criteria!r}")
print(f"Text     : {SPAN_OUTPUT!r}")
print(f"Choices  : {eval_config.choices}")

In [ ]:
# ── Build input summary manually (no DB) ────────────────────────────────────
# This is what build_input_summary() returns for a real span.
input_summary = "\n".join([
    f"SPAN: customer_message_eval (type: llm)",
    f"Status: success",
    f"Model: gpt-4",
    f"Input preview:  {SPAN_INPUT}",
    f"Output preview: {SPAN_OUTPUT}",
    f"Has metadata: False",
])

available_resources = eval_config.available_resources

print("Input Summary (sent to Scout):")
print("─" * 55)
print(input_summary)

---
## Stage 1 — Scout

Cheap Haiku pre-pass. Reads criteria + input summary + resource list → produces `EvalBrief`.

In [ ]:
print("=" * 60)
print("  STAGE 1: SCOUT  (TURING_FLASH — cheap pre-pass)")
print("=" * 60)

scout = EvalScout()
brief = scout.run(
    criteria           = eval_config.criteria,
    input_summary      = input_summary,
    available_resources= available_resources,
)

print()
print("Scout Brief:")
print(json.dumps(brief, indent=2))

scout_tokens = scout.get_token_usage()
print()
print(f"Scout token usage: {scout_tokens}")

In [ ]:
# Attach resources + token usage to the brief before handing to the Orchestrator
brief["available_resources"]  = available_resources
brief["_scout_token_usage"]   = scout_tokens

print("Brief summary:")
print(f"  complexity                   : {brief['complexity']}")
print(f"  recommended_model            : {brief['recommended_model']}")
print(f"  criteria_needs_formalization : {brief['criteria_needs_formalization']}")
print(f"  data_plan.sufficient         : {brief['data_plan']['sufficient']}")
print(f"  resource_plan.use_kb         : {brief['resource_plan']['use_kb']}")
print(f"  resource_plan.use_feedback   : {brief['resource_plan']['use_feedback']}")
print(f"  reasoning:")
for line in textwrap.wrap(brief.get('reasoning', ''), width=70):
    print(f"    {line}")

---
## Stage 2 — Orchestrator

Sonnet agentic loop.  
`inspect_data` is monkey-patched to return the test span (no real DB).  
`_execute_tool` is wrapped to print every tool call and result.

In [ ]:
orchestrator = EvalOrchestrator(
    project_id     = eval_config.project_id,
    eval_config    = eval_config,
    input_resolver = InputResolver(),
    scout_brief    = brief,
)

print("Dynamic tool list built by Orchestrator:")
for t in orchestrator.tools:
    print(f"  • {t['function']['name']}")

In [ ]:
# ── Patch 1: mock inspect_data so no real DB is needed ──────────────────────
_real_execute_tool = orchestrator._execute_tool.__func__  # unbound

def _mock_inspect_data(self, target_type, target_id, depth):
    """Return the test span content without hitting the DB."""
    if depth == "detail":
        self._drill_down_performed = True
    return {
        "span_id"   : target_id,
        "name"      : "customer_message_eval",
        "type"      : "llm",
        "status"    : "success",
        "input"     : SPAN_INPUT,
        "output"    : SPAN_OUTPUT,
        "metadata"  : {},
        "model"     : "gpt-4",
        "latency_ms": 380,
    }

orchestrator._tool_inspect_data = types.MethodType(_mock_inspect_data, orchestrator)
print("✓ inspect_data patched (no DB)")

# ── Patch 2: wrap _execute_tool to print every tool call ────────────────────
_real_execute = orchestrator._execute_tool

_turn_counter = [0]   # mutable counter shared by closure

def _verbose_execute_tool(tool_name, args):
    _turn_counter[0] += 1
    sep = "─" * 55

    print(f"\n{sep}")
    print(f"  TOOL CALL #{_turn_counter[0]}: {tool_name}")
    print(sep)

    # Pretty-print args, truncating long strings
    def _trim(v, n=300):
        s = str(v)
        return s[:n] + " …" if len(s) > n else s

    for k, v in args.items():
        print(f"  {k}: {_trim(v)}")

    result = _real_execute(tool_name, args)

    print(f"  ↳ result:")
    result_str = json.dumps(result, default=str, ensure_ascii=False)
    for line in textwrap.wrap(result_str, width=80):
        print(f"    {line}")

    return result

orchestrator._execute_tool = _verbose_execute_tool
print("✓ _execute_tool patched (verbose tracing)")

In [ ]:
print("=" * 60)
print("  STAGE 2: ORCHESTRATOR  (TURING_SMALL — agentic loop)")
print("=" * 60)
print(f"  Max turns   : {orchestrator.MAX_TURNS}")
print(f"  Max run_eval: {orchestrator.MAX_EVAL_CALLS}")
print()

orch_result = orchestrator.run()

print()
print("=" * 60)
print("  ORCHESTRATOR DONE")
print("=" * 60)

---
## Final Results

In [ ]:
eval_result = orch_result["result"]

print("━" * 60)
print("  EVAL RESULT")
print("━" * 60)
print(f"  Result      : {eval_result.get('result')}")
print(f"  Confidence  : {eval_result.get('confidence'):.2f}")
print(f"  Model used  : {orch_result.get('model_used')}")
print(f"  Eval calls  : {orch_result.get('eval_calls')}")
print(f"  Orch turns  : {orch_result.get('orchestrator_turns')}")
print(f"  Drill-down  : {orch_result.get('drill_down_performed')}")
print(f"  Resources   : {orch_result.get('resources_used')}")
print()
print("  Explanation:")
for line in textwrap.wrap(eval_result.get('explanation', ''), width=70):
    print(f"    {line}")

print()
print("━" * 60)
print("  TOKEN USAGE")
print("━" * 60)
tu = orch_result.get("token_usage", {})
for stage, counts in tu.items():
    if isinstance(counts, dict):
        p = counts.get('prompt_tokens', 0)
        c = counts.get('completion_tokens', 0)
        print(f"  {stage:<18} prompt={p:>6}  completion={c:>6}  total={p+c:>6}")
    else:
        print(f"  {stage}: {counts}")